<a href="https://colab.research.google.com/github/ryukf333/remote-sensing-crop-yield-ml/blob/main/03sentinel_features.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import ee
import geemap
import pandas as pd
from google.colab import drive
import os

drive.mount('/content/drive')
ee.Authenticate()
ee.Initialize(project='ee-crop-yield-2026')

# Function to calculate NDVI
def add_ndvi(image):
    # Sentinel-2 NDVI formula: (NIR - Red) / (NIR + Red)
    # For Sentinel-2 SR: NIR is B8, Red is B4
    ndvi = image.normalizedDifference(['B8', 'B4']).rename('NDVI')
    return image.addBands(ndvi)

Mounted at /content/drive


*** Earth Engine *** Share your feedback by taking our Annual Developer Satisfaction Survey: https://google.qualtrics.com/jfe/form/SV_9oS0DRcPvElRMNw?source=python


In [ ]:
# 1. Load Iowa counties and the 2022 Corn Mask
counties = ee.FeatureCollection("TIGER/2018/Counties").filter(ee.Filter.eq('STATEFP', '19'))
corn_mask = ee.Image("USDA/NASS/CDL/2022").select('cropland').eq(1)

# 2. Filter Sentinel-2 Surface Reflectance for July 2022 over Iowa
s2_july = ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED") \
    .filterBounds(counties) \
    .filterDate('2022-07-01', '2022-07-31') \
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20)) # Keep mostly clear images

# 3. Apply the NDVI calculation and create a single median composite for the month
s2_ndvi_july = s2_july.map(add_ndvi).select('NDVI').median()

# 4. CRITICAL STEP: Mask the satellite image so it ONLY looks at corn fields
s2_corn_only = s2_ndvi_july.updateMask(corn_mask)

# 5. Calculate the average NDVI for each county
county_ndvi = s2_corn_only.reduceRegions(
    collection=counties,
    reducer=ee.Reducer.mean(),
    scale=30 # 30m resolution to match the CDL
)

# 6. Convert to Pandas DataFrame
df_ndvi = geemap.ee_to_df(county_ndvi)

# Clean columns for the master table
df_ndvi['county_fips'] = df_ndvi['COUNTYFP'].astype(str).str.zfill(3)
df_ndvi['state_fips'] = df_ndvi['STATEFP'].astype(str).str.zfill(2)
df_ndvi = df_ndvi[['state_fips', 'county_fips', 'NAME', 'mean']].rename(columns={'mean': 'ndvi_mean_jul', 'NAME': 'county'})

# 7. Save to Drive
out_path = '/content/drive/MyDrive/remote-sensing-crop-yield-ml/data_processed/iowa_ndvi_jul_2022.csv'
df_ndvi.to_csv(out_path, index=False)

print(f"Extracted Sentinel-2 NDVI for {len(df_ndvi)} counties.")
display(df_ndvi.head())

Extracted Sentinel-2 NDVI for 99 counties.


,state_fips,county_fips,county,ndvi_mean_jul
0,19,149,Plymouth,0.861995
1,19,193,Woodbury,0.860499
2,19,027,Carroll,0.873902
3,19,061,Dubuque,0.870410
4,19,101,Jefferson,0.878176


# extract weather data

In [ ]:
# 1. Load Iowa county boundaries
counties = ee.FeatureCollection("TIGER/2018/Counties").filter(ee.Filter.eq('STATEFP', '19'))

# 2. Load the ERA5-Land Monthly Aggregated dataset for July 2022
era5_july = ee.ImageCollection("ECMWF/ERA5_LAND/MONTHLY_AGGR") \
    .filterDate('2022-07-01', '2022-07-31') \
    .first() # Grab the single monthly image

# 3. Select Temperature and Soil Moisture bands
# Temperature is in Kelvin; Soil water is volumetric (m3/m3)
weather_bands = era5_july.select(['temperature_2m', 'volumetric_soil_water_layer_1'])

# 4. Calculate the average weather for each county polygon
county_weather = weather_bands.reduceRegions(
    collection=counties,
    reducer=ee.Reducer.mean(),
    scale=9000 # ERA5-Land native resolution is ~9km
)

# 5. Convert to a Pandas DataFrame
df_weather = geemap.ee_to_df(county_weather)

# 6. Clean the columns for our master index
df_weather['county_fips'] = df_weather['COUNTYFP'].astype(str).str.zfill(3)
df_weather['state_fips'] = df_weather['STATEFP'].astype(str).str.zfill(2)

# Select core columns and rename them for clarity
df_weather = df_weather[['state_fips', 'county_fips', 'NAME', 'temperature_2m', 'volumetric_soil_water_layer_1']]
df_weather = df_weather.rename(columns={
    'NAME': 'county',
    'temperature_2m': 'temp_k_jul',
    'volumetric_soil_water_layer_1': 'soil_water_jul'
})

# 7. Save to the processed data folder
out_path = '/content/drive/MyDrive/remote-sensing-crop-yield-ml/data_processed/iowa_weather_jul_2022.csv'
df_weather.to_csv(out_path, index=False)

print(f"Extracted ERA5-Land weather for {len(df_weather)} counties.")
display(df_weather.head())

Extracted ERA5-Land weather for 99 counties.


,state_fips,county_fips,county,temp_k_jul,soil_water_jul
0,19,149,Plymouth,298.587972,0.118269
1,19,193,Woodbury,298.486085,0.189004
2,19,027,Carroll,297.388217,0.242190
3,19,061,Dubuque,296.587366,0.296065
4,19,101,Jefferson,298.886344,0.195399


# Data Merge

In [ ]:
processed_dir = '/content/drive/MyDrive/remote-sensing-crop-yield-ml/data_processed'

In [ ]:
df_yield = pd.read_csv(os.path.join(processed_dir, 'iowa_yield_2018_2022.csv'))
df_yield_2022 = df_yield[df_yield['year'] == 2022].copy()

# Predictors
df_ndvi = pd.read_csv(os.path.join(processed_dir, 'iowa_ndvi_jul_2022.csv'))
df_weather = pd.read_csv(os.path.join(processed_dir, 'iowa_weather_jul_2022.csv'))

# Ensure the FIPS codes are strings formatted with leading zeros for merging
for df in [df_yield_2022, df_ndvi, df_weather]:
    df['state_fips'] = df['state_fips'].astype(str).str.zfill(2)
    df['county_fips'] = df['county_fips'].astype(str).str.zfill(3)

print("Datasets loaded successfully.")

Datasets loaded successfully.


In [ ]:
# 1. Start with the yield spine and merge NDVI
merged_df = pd.merge(
    df_yield_2022[['state_fips', 'county_fips', 'county', 'year', 'yield_bu_ac']],
    df_ndvi[['state_fips', 'county_fips', 'ndvi_mean_jul']],
    on=['state_fips', 'county_fips'],
    how='inner' # Only keep counties that exist in both tables
)

# 2. Merge Weather
final_df = pd.merge(
    merged_df,
    df_weather[['state_fips', 'county_fips', 'temp_k_jul', 'soil_water_jul']],
    on=['state_fips', 'county_fips'],
    how='inner'
)

# 3. Data Leakage & Missingness Audit
print("--- Data Quality Audit ---")
print(f"Total Rows: {len(final_df)}")
print("\nMissing Values per Feature:")
print(final_df.isnull().sum())

# Ensure all data in this table is strictly from 2022 to prevent cross-year leakage
unique_years = final_df['year'].unique()
print(f"\nUnique Years in Table: {unique_years} (Should only be [2022])")

display(final_df.head())

--- Data Quality Audit ---
Total Rows: 97

Missing Values per Feature:
state_fips        0
county_fips       0
county            0
year              0
yield_bu_ac       0
ndvi_mean_jul     2
temp_k_jul        0
soil_water_jul    0
dtype: int64

Unique Years in Table: [2022] (Should only be [2022])


,state_fips,county_fips,county,year,yield_bu_ac,ndvi_mean_jul,temp_k_jul,soil_water_jul
0,19,001,ADAIR,2022,164.2,0.817093,297.745176,0.263439
1,19,005,ALLAMAKEE,2022,215.9,0.848514,296.075046,0.329033
2,19,007,APPANOOSE,2022,146.1,0.880191,299.053178,0.211331
3,19,009,AUDUBON,2022,197.9,0.868926,297.237520,0.275350
4,19,011,BENTON,2022,221.7,0.852600,297.152621,0.242012


In [ ]:
# 1. Identify which counties have missing NDVI
missing_counties = final_df[final_df['ndvi_mean_jul'].isna()]
print("Counties missing Sentinel-2 NDVI in July 2022:")
display(missing_counties[['state_fips', 'county_fips', 'county', 'yield_bu_ac']])

# 2. Drop the missing rows
final_df_clean = final_df.dropna(subset=['ndvi_mean_jul']).copy()
print(f"\nFinal dataset ready for modeling: {len(final_df_clean)} rows.")

# 3. Overwrite the saved master modeling table with the clean data
out_path = '/content/drive/MyDrive/remote-sensing-crop-yield-ml/data_processed/iowa_master_modeling_2022.csv'
final_df_clean.to_csv(out_path, index=False)
print("Saved clean master table to Drive!")

Counties missing Sentinel-2 NDVI in July 2022:


,state_fips,county_fips,county,yield_bu_ac
7,19,017,BREMER,210.8
17,19,037,CHICKASAW,214.6



Final dataset ready for modeling: 95 rows.
Saved clean master table to Drive!
